In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import StratifiedKFold

In [ ]:
df = pd.read_csv("train.csv")

X = df.drop(columns=["project_id", "hours_spent", "log_hours", "hours_bucket"])
y_log = np.log1p(df["hours_spent"])

print(f"Rows: {len(df):,}  Features: {X.shape[1]}")

Rows: 4,500  Features: 69


In [ ]:
PRIOR_MAP = {"0": 0, "1-2": 1, "3-5": 2, "6+": 3}
AGE_MAP = {"<1": 0, "1-2": 1, "3-5": 2, "6+": 3}
BOOL_MAP = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}

CAT_COLS = ["primary_language", "framework", "hosting_type", "commit_cadence", "project_category"]
BOOL_COLS = [
    "uses_game_engine", "has_tests", "has_ci_config", "has_dockerfile", "has_backend",
    "has_database", "has_custom_art", "readme_has_install_instructions", "has_license",
    "has_docs_folder", "url_is_live", "is_mobile_responsive", "has_custom_domain",
    "is_multiplayer", "mentions_ai_api", "is_clone_or_tutorial", "is_forked_from_template",
]


def clean(X):
    X = X.copy()
    X["author_prior_projects"] = X["author_prior_projects"].astype(str).map(PRIOR_MAP)
    X["author_account_age_years"] = X["author_account_age_years"].astype(str).map(AGE_MAP)
    for c in BOOL_COLS:
        if c in X.columns:
            X[c] = X[c].map(BOOL_MAP)
    for c in CAT_COLS:
        X[c] = X[c].fillna("missing").astype(str)
    return X


def div(a, b):
    return a / b.replace(0, np.nan)


def add_features(X):
    X = X.copy()
    X["lines_per_active_day"] = div(X.lines_added_total, X.active_commit_days)
    X["loc_per_commit"] = div(X.lines_of_code, X.num_commits)
    X["commits_per_active_day"] = div(X.num_commits, X.active_commit_days)
    X["active_frac"] = div(X.active_commit_days, X.project_span_days + 1)
    X["total_lines_touched"] = np.log1p(X.lines_added_total + X.lines_deleted_total)
    X["biggest_commit_share"] = div(X.max_lines_in_one_commit, X.lines_added_total)
    X["bulk_upload"] = (X.pct_code_in_first_commit > 0.8).astype(float).where(X.pct_code_in_first_commit.notna())
    X["few_commits"] = (X.num_commits <= 3).astype(float).where(X.num_commits.notna())
    X["no_git"] = X.num_commits.isna().astype(float)
    X["n_assets"] = X[["num_image_files", "num_audio_files", "num_3d_model_files", "num_font_files"]].sum(axis=1, min_count=1)
    X["readme_richness"] = X[["readme_num_headings", "readme_num_images", "readme_num_code_blocks"]].sum(axis=1, min_count=1)
    X["loc_x_days"] = np.log1p(X.lines_of_code) * np.log1p(X.active_commit_days)
    X["loc_template_adj"] = np.log1p(X.lines_of_code) * (1 - 0.5 * X.is_forked_from_template.fillna(0))

    loc = np.log1p(X.lines_of_code)
    first = X.pct_code_in_first_commit.fillna(0)
    X["assets_per_kloc"] = np.log1p(X.n_assets.fillna(0)) - np.log1p(X.lines_of_code / 1000)
    X["asset_mb_log"] = np.log1p(X.asset_size_mb)
    X["dump_size"] = loc * first          # code pasted in at once
    X["written_loc"] = loc * (1 - first)  # code written over time
    X["site_size"] = np.log1p(X.num_internal_pages) + np.log1p(X.page_weight_kb)
    X["work_days_x_assets"] = np.log1p(X.active_commit_days) * np.log1p(X.n_assets.fillna(0))
    X["span_weeks_active"] = np.log1p(X.project_span_days / 7) * X.active_frac.fillna(0)
    return X


Xf = add_features(clean(X))
print(f"Features after engineering: {Xf.shape[1]}")

Features after engineering: 89


In [ ]:
SEEDS = [0, 1, 2, 3, 4]
LEARNING_RATE = 0.01
ITERATIONS = 5000

MODEL_PARAMS = {
    "cat_features": CAT_COLS,
    "iterations": ITERATIONS,
    "learning_rate": LEARNING_RATE,
    "depth": 8,
    "l2_leaf_reg": 5,
    "rsm": 0.5,
    "loss_function": "RMSE",
    "allow_writing_files": False,
    "verbose": 0,
}

In [8]:
import time
CV_SEEDS = [0, 1, 2]
CV_PARAMS = {**MODEL_PARAMS, "learning_rate": 0.03, "iterations": 1800}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(Xf))
oof_single = np.zeros(len(Xf))
fold_scores = []

n_total = 5 * len(CV_SEEDS)
n_done = 0
start = time.time()

for fold, (tr, va) in enumerate(cv.split(Xf, df["hours_bucket"]), 1):
    seed_preds = []
    for seed in CV_SEEDS:
        m = CatBoostRegressor(random_seed=seed, **CV_PARAMS)
        m.fit(Xf.iloc[tr], y_log.iloc[tr])
        seed_preds.append(m.predict(Xf.iloc[va]))

        n_done += 1
        per_model = (time.time() - start) / n_done
        print(f"  fold {fold} seed {seed}: {per_model:.0f}s/model, "
              f"~{per_model * (n_total - n_done) / 60:.1f} min left")

    oof[va] = np.mean(seed_preds, axis=0)
    oof_single[va] = seed_preds[0]
    fold_scores.append(root_mean_squared_error(y_log.iloc[va], oof[va]))
    print(f"fold {fold}: {fold_scores[-1]:.5f}")

print(f"\nMean CV: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")
print(f"OOF, {len(CV_SEEDS)}-seed average: {root_mean_squared_error(y_log, oof):.5f}")
print(f"OOF, seed 0 only:    {root_mean_squared_error(y_log, oof_single):.5f}")
print(f"Total: {(time.time() - start) / 60:.1f} min")

  fold 1 seed 0: 77s/model, ~18.1 min left
  fold 1 seed 1: 76s/model, ~16.5 min left
  fold 1 seed 2: 74s/model, ~14.8 min left
fold 1: 0.64446
  fold 2 seed 0: 73s/model, ~13.3 min left
  fold 2 seed 1: 76s/model, ~12.6 min left
  fold 2 seed 2: 75s/model, ~11.2 min left
fold 2: 0.67057
  fold 3 seed 0: 74s/model, ~9.9 min left
  fold 3 seed 1: 74s/model, ~8.6 min left
  fold 3 seed 2: 73s/model, ~7.3 min left
fold 3: 0.64653
  fold 4 seed 0: 73s/model, ~6.1 min left
  fold 4 seed 1: 73s/model, ~4.8 min left
  fold 4 seed 2: 72s/model, ~3.6 min left
fold 4: 0.66921
  fold 5 seed 0: 72s/model, ~2.4 min left
  fold 5 seed 1: 72s/model, ~1.2 min left
  fold 5 seed 2: 72s/model, ~0.0 min left
fold 5: 0.65193

Mean CV: 0.65654 +/- 0.01118
OOF, 3-seed average: 0.65664
OOF, seed 0 only:    0.65744
Total: 18.0 min


In [9]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

final_models = []
for seed in SEEDS:
    m = CatBoostRegressor(random_seed=seed, **MODEL_PARAMS)
    m.fit(Xf, y_log)
    path = MODEL_DIR / f"model_seed{seed}.cbm"
    m.save_model(str(path))
    final_models.append(m)
    print(f"saved {path} ({path.stat().st_size / 1e6:.1f} MB)")

saved models/model_seed0.cbm (21.2 MB)
saved models/model_seed1.cbm (21.2 MB)
saved models/model_seed2.cbm (21.2 MB)
saved models/model_seed3.cbm (21.2 MB)
saved models/model_seed4.cbm (21.2 MB)


In [10]:
importance = pd.concat(
    [m.get_feature_importance(prettified=True).set_index("Feature Id")["Importances"] for m in final_models],
    axis=1,
).mean(axis=1).sort_values(ascending=False)

print(importance.head(25).round(2).to_string())

Feature Id
description_word_count      6.06
loc_x_days                  4.72
span_weeks_active           4.65
median_lines_per_commit     3.25
active_commit_days          3.07
hosting_type                3.00
primary_language            2.98
loc_template_adj            2.82
pct_generic_commit_msgs     2.48
project_category            2.35
num_source_files            2.14
pct_commits_weekend         2.07
readme_word_count           1.95
biggest_commit_share        1.91
commits_per_active_day      1.91
avg_commit_msg_length       1.90
pct_primary_language        1.87
pct_commits_late_night      1.87
lines_added_total           1.84
comment_line_ratio          1.79
total_lines_touched         1.75
avg_file_length             1.64
pct_code_in_first_commit    1.60
project_span_days           1.59
lines_of_code               1.58
